# Object Detection with GeoAI

GeoAI's `train_detection()` trains a YOLO-based object detector on georeferenced imagery with automatic chip generation, training, and tiled inference. Results are returned as georeferenced GeoJSON and can be visualized on an interactive Leafmap map.

## References
- GeoAI docs: https://geoai.gishub.org/
- Ultralytics YOLO: https://docs.ultralytics.com/

## 1. Setup

In [ ]:
import os
import urllib.request
import geoai
import matplotlib.pyplot as plt

# Download a sample aerial image with labeled objects
os.makedirs("detection_data", exist_ok=True)
image_url, label_url = geoai.data.get_vehicle_sample_urls()

image_path = "detection_data/image.tif"
label_path = "detection_data/labels.geojson"  # GeoJSON with bounding box annotations

if not os.path.exists(image_path):
    urllib.request.urlretrieve(image_url, image_path)
if not os.path.exists(label_path):
    urllib.request.urlretrieve(label_url, label_path)

print(f"Data ready in detection_data/")

## 2. Prepare YOLO Dataset

In [ ]:
from geoai import prepare_detection_dataset

prepare_detection_dataset(
    image=image_path,
    labels=label_path,
    output_dir="yolo_data",
    chip_size=640,
    stride=320,
    train_split=0.8,
    format="yolo",
)

print("YOLO dataset prepared in yolo_data/")

## 3. Train a YOLO Detector

In [ ]:
from geoai import train_detection

model = train_detection(
    data_yaml="yolo_data/data.yaml",
    model_size="n",       # 'n' (nano), 's', 'm', 'l', 'x'
    epochs=50,
    imgsz=640,
    batch=8,
    output_dir="detection_output",
    device="cpu",         # change to 0 for GPU
)

print("Training complete.")

## 4. Run Tiled Inference

In [ ]:
from geoai import predict_detection

detections_gdf = predict_detection(
    model=model,
    image=image_path,
    output="detections.geojson",
    chip_size=640,
    overlap=0.2,
    confidence=0.3,
    nms_iou=0.45,
)

print(f"Detected {len(detections_gdf)} objects")
print(detections_gdf.head())

## 5. Visualize Results

In [ ]:
import numpy as np
import rasterio

with rasterio.open(image_path) as src:
    img_array = src.read([1, 2, 3])
    img_array = np.moveaxis(img_array, 0, -1)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(img_array, extent=extent)
detections_gdf.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=1.5)
ax.set_title(f"GeoAI object detection — {len(detections_gdf)} objects")
plt.tight_layout()
plt.show()

## 6. Interactive Map with Leafmap

GeoAI integrates with Leafmap for browser-based interactive visualization. Run this cell in JupyterLab.

In [ ]:
import leafmap

m = leafmap.Map()
m.add_raster(image_path, layer_name="Aerial image")
m.add_geojson("detections.geojson", layer_name="Detections", style={"color": "red", "weight": 2})
m